# PGS Tangency Radii

This notebook shows how to calculate the tangency radii for Gaussians with normals over a mesh surface.

In [1]:
import torch
import numpy as np
import plotly.graph_objects as go
import math
import trimesh
from geocutool.primitive.gs import compute_gs_covi, compute_gs_aabb
from geocutool.primitive.pgs import solve_pgs_cluster_tangency_radius

## 1. Visualization Helpers

Define helper functions to generate wireframes for voxel bounding boxes.

In [2]:
def generate_batch_wireframe(mins, maxs):
    """
    Takes N voxel min/max tensors, returns single flattened lists of x, y, z coords
    with None separators to draw all boxes in a single Plotly trace.
    """
    x_lines, y_lines, z_lines = [], [], []
    
    # Convert to numpy if they are tensors
    mins_np = mins.cpu().numpy()
    maxs_np = maxs.cpu().numpy()
    
    for min_pt, max_pt in zip(mins_np, maxs_np):
        # 8 corners of the voxel
        xc = [
            min_pt[0], max_pt[0], max_pt[0], min_pt[0], 
            min_pt[0], max_pt[0], max_pt[0], min_pt[0]]
        yc = [
            min_pt[1], min_pt[1], max_pt[1], max_pt[1], 
            min_pt[1], min_pt[1], max_pt[1], max_pt[1]]
        zc = [
            min_pt[2], min_pt[2], min_pt[2], min_pt[2], 
            max_pt[2], max_pt[2], max_pt[2], max_pt[2]]

        # The 12 edge drawing path (with Nones to lift the pen)
        x_lines.extend([
            xc[0], xc[1], xc[2], xc[3], xc[0], None, 
            xc[4], xc[5], xc[6], xc[7], xc[4], None, 
            xc[0], xc[4], None, xc[1], xc[5], None, 
            xc[2], xc[6], None, xc[3], xc[7], None])
        y_lines.extend([
            yc[0], yc[1], yc[2], yc[3], yc[0], None, 
            yc[4], yc[5], yc[6], yc[7], yc[4], None, 
            yc[0], yc[4], None, yc[1], yc[5], None, 
            yc[2], yc[6], None, yc[3], yc[7], None])
        z_lines.extend([
            zc[0], zc[1], zc[2], zc[3], zc[0], None, 
            zc[4], zc[5], zc[6], zc[7], zc[4], None, 
            zc[0], zc[4], None, zc[1], zc[5], None, 
            zc[2], zc[6], None, zc[3], zc[7], None])
        
    return x_lines, y_lines, z_lines

## 2. Mesh Definition

Define the mesh geometry (vertices and faces) where Gaussians are sampled.

In [3]:
vertices = [
    [-1, -1, -0.8],
    [-1, 1, -0.8],
    [1, -1, -0.8],
    [1, 1, -0.8],
    [-1, 0, 0],
    [1, 0, 0]
]

faces = [
    [0, 1, 2],
    [1, 3, 2],
    [0, 5, 4],
    [0, 2, 5],
    [4, 3, 1],
    [4, 5, 3],
    [2, 3, 5],
    [0, 4, 1],
]

vertices = np.array(vertices)

num_points = 10000

mesh = trimesh.Trimesh(vertices=vertices, faces=faces)

points, face_index = trimesh.sample.sample_surface(mesh, num_points)
normals = mesh.face_normals[face_index]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

means = torch.from_numpy(points).float().to(device)
gs_normals = torch.from_numpy(normals).float().to(device)
rotations = torch.rand(means.shape[0], 4).float().to(device)
scales = torch.ones(means.shape[0], 3).float().to(device) * 0.05
opacities = torch.rand(means.shape[0]).float().to(device)

print("Means shape:", means.shape)
print("Normals shape:", gs_normals.shape)
print("Rotations shape:", rotations.shape)
print("Scales shape:", scales.shape)
print(f"Scales mean: {scales.mean().item():.4f}, std: {scales.std().item():.4f}")
print("Opacities shape:", opacities.shape)

Means shape: torch.Size([10000, 3])
Normals shape: torch.Size([10000, 3])
Rotations shape: torch.Size([10000, 4])
Scales shape: torch.Size([10000, 3])
Scales mean: 0.0500, std: 0.0000
Opacities shape: torch.Size([10000])


## 3. Mesh Voxelization

Create an Octree representation of the mesh to efficiently query surface locations.

In [4]:
res = 16

max_level = int(math.log2(res))
min_level = min(4, max(1, max_level - 1))

level = max_level
iso = 0.1
tol = 1. / 8.
rotnorm = True
k=25

covis = compute_gs_covi(means, rotations, scales, level, tol, rotnorm)
print("Covariance Inverses shape:", covis.shape)

isos, invalid_mask = solve_pgs_cluster_tangency_radius(means, gs_normals, covis, k)
print("Tangency Radii shape:", isos.shape)
print("Invalid Mask shape:", invalid_mask.shape)
print(f"Num invalid: {invalid_mask.sum().item()}, valid: {(~invalid_mask).sum().item()}")

isos_min = isos[~invalid_mask].min()
isos_mean = isos[~invalid_mask].mean()
isos_max = isos[~invalid_mask].max()

print(f"Isos mean: {isos_mean.item():.6f}, min: {isos_min.item():.6f}, max: {isos_max.item():.6f}")

isos[invalid_mask] = isos_max

print(f"Isos mean: {isos.mean().item():.6f}, min: {isos.min().item():.6f}, max: {isos.max().item():.6f}")

aabb_min, aabb_max, contact_points = compute_gs_aabb(means, scales, covis, level, isos, tol)

print("AABB Min:", aabb_min.shape)
print("AABB Max:", aabb_max.shape)
print("Contact Points:", contact_points.shape)
print("Gaussian Normals:", gs_normals.shape)

means_np = means.cpu().numpy()
contact_points_np = contact_points.cpu().numpy()
rot_np = rotations.cpu().numpy()
scales_np = scales.cpu().numpy()
normals_np = gs_normals.cpu().numpy()
covis_np = covis.cpu().numpy()
isos_np = isos.cpu().numpy()

Covariance Inverses shape: torch.Size([10000, 6])
Tangency Radii shape: torch.Size([10000])
Invalid Mask shape: torch.Size([10000])
Num invalid: 7322, valid: 2678
Isos mean: 1.732514, min: 0.000000, max: 11.367575
Isos mean: 8.787305, min: 0.000000, max: 11.367575
AABB Min: torch.Size([10000, 3])
AABB Max: torch.Size([10000, 3])
Contact Points: torch.Size([10000, 9])
Gaussian Normals: torch.Size([10000, 3])


## 4. Quaternion Utilities

Define helper functions for converting quaternions into rotation matrices.

In [5]:
def quat_to_rotmat(q):
    """Converts a quaternion [w, x, y, z] to a 3x3 rotation matrix."""
    w, x, y, z = q[0], q[1], q[2], q[3]
    n = np.sqrt(x*x + y*y + z*z + w*w)
    w, x, y, z = w/n, x/n, y/n, z/n
    
    R = np.array([
        [1 - 2*(y**2 + z**2), 2*(x*y - w*z),     2*(x*z + w*y)],
        [2*(x*y + w*z),     1 - 2*(x**2 + z**2), 2*(y*z - w*x)],
        [2*(x*z - w*y),     2*(y*z + w*x),     1 - 2*(x**2 + y**2)]
    ])
    return R

def get_ellipsoid_surface(mean, rot_quat, scale, iso_sq, resolution=15):
    """Generates X, Y, Z grid for a 3D ellipsoid surface."""
    # The mathematical distance multiplier is the square root of the squared ISO
    multiplier = np.sqrt(iso_sq)
    
    # 1. Base Unit Sphere
    u = np.linspace(0, 2 * np.pi, resolution)
    v = np.linspace(0, np.pi, resolution)
    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones_like(u), np.cos(v))
    
    # 2. Scale the sphere by (scale * sqrt(iso))
    x *= scale[0] * multiplier
    y *= scale[1] * multiplier
    z *= scale[2] * multiplier
    
    # 3. Rotate the scaled points
    R = quat_to_rotmat(rot_quat)
    points = np.stack([x.flatten(), y.flatten(), z.flatten()], axis=1)
    points_rot = points @ R.T  # Apply rotation
    
    # 4. Translate to the Gaussian mean
    x_rot = points_rot[:, 0].reshape(resolution, resolution) + mean[0]
    y_rot = points_rot[:, 1].reshape(resolution, resolution) + mean[1]
    z_rot = points_rot[:, 2].reshape(resolution, resolution) + mean[2]
    
    return x_rot, y_rot, z_rot

## 5. Radii Visualization

Visualize the projected bounding boxes on the mesh surface, using the computed tangency radii.

In [ ]:
num_to_plot = 100

plot_mins = aabb_min[:num_to_plot]
plot_maxs = aabb_max[:num_to_plot]
plot_means = means_np[:num_to_plot]
plot_normals = normals_np[:num_to_plot]
plot_scales = scales_np[:num_to_plot]
plot_rots = rot_np[:num_to_plot]
plot_isos = isos_np[:num_to_plot]

# Reshape contact points to standard (X, Y, Z) array
plot_contacts = contact_points_np[:num_to_plot].reshape(-1, 3)

# Generate Wireframes
x_lines, y_lines, z_lines = generate_batch_wireframe(plot_mins, plot_maxs)

# Mesh
vertices = mesh.vertices
faces = mesh.faces

# Normals
nx_lines, ny_lines, nz_lines = [], [], []
normal_length = 0.5

for i in range(num_to_plot):
    nx_lines.extend([plot_means[i, 0], plot_means[i, 0] + plot_normals[i, 0] * normal_length, None])
    ny_lines.extend([plot_means[i, 1], plot_means[i, 1] + plot_normals[i, 1] * normal_length, None])
    nz_lines.extend([plot_means[i, 2], plot_means[i, 2] + plot_normals[i, 2] * normal_length, None])

# Start adding traces to the figure
fig = go.Figure()

# Mesh
fig.add_trace(go.Mesh3d(
    x=vertices[:, 0],
    y=vertices[:, 1],
    z=vertices[:, 2],
    i=faces[:, 0],
    j=faces[:, 1],
    k=faces[:, 2],
    color='lightblue',
    opacity=0.8,
    name='Original Mesh'
))

# Add Gaussian Normals
fig.add_trace(go.Scatter3d(
    x=nx_lines, y=ny_lines, z=nz_lines,
    mode='lines',
    line=dict(color='purple', width=4),
    name='Gaussian Normals'
))

# Add AABB Wireframes
# fig.add_trace(go.Scatter3d(
#     x=x_lines, y=y_lines, z=z_lines,
#     mode='lines', line=dict(color='lightgreen', width=2),
#     name='AABBs'
# ))

# Add Gaussian Centers
# fig.add_trace(go.Scatter3d(
#     x=plot_means[:, 0], y=plot_means[:, 1], z=plot_means[:, 2],
#     mode='markers', marker=dict(size=4, color='blue', symbol='circle'),
#     name='Gaussian Centers'
# ))

# Add Contact Points (Should lie perfectly on the wireframe and the ellipsoid!)
# fig.add_trace(go.Scatter3d(
#     x=plot_contacts[:, 0], y=plot_contacts[:, 1], z=plot_contacts[:, 2],
#     mode='markers', marker=dict(size=4, color='red', symbol='diamond'),
#     name='Contact Points'
# ))

# --- Loop through to generate and plot the 3D Ellipsoids ---
for i in range(num_to_plot):
    ex, ey, ez = get_ellipsoid_surface(
        mean=plot_means[i], 
        rot_quat=plot_rots[i], 
        scale=plot_scales[i], 
        iso_sq=plot_isos[i]
    )
    
    fig.add_trace(go.Surface(
        x=ex, y=ey, z=ez,
        opacity=0.2,
        colorscale='Viridis',
        showscale=False,
        hoverinfo='skip'
    ))

# Render
fig.update_layout(
    title=f"First {num_to_plot} Gaussians: Ellipsoids, AABBs & Contact Points",
    scene=dict(
        aspectmode='data', # Critical: Ensures X/Y/Z scaling is exactly 1:1:1
        xaxis=dict(title='X'),
        yaxis=dict(title='Y'),
        zaxis=dict(title='Z')
    )
)
fig.show()